## 1. 📉 Model Biaya Produksi & Titik Impas (BEP)

### 📚 Perhitungan Analitik:
**Diketahui (Given):**
- Biaya Tetap ($FC$) = Rp 100.000 (Sewa tempat, alat, dll)
- Biaya Variabel per unit ($VC$) = Rp 5.500 (Bahan ubi, susu, cup)
- Harga Jual per unit ($p$) = Rp 10.000

**Ditanya (Asked):**
- Berapa jumlah unit ($q$) yang harus terjual agar mencapai Titik Impas (BEP)?

**Dijawab (Answered):**
Menggunakan rumus integral dasar dari fungsi marginal atau rumus aljabar BEP:
$$q_{BEP} = \frac{FC}{p - VC}$$

---
## ⚙️ Setup Library

In [1]:
import numpy as np
import pandas as pd
import sympy as sp
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import integrate, optimize
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False
})

print('✅ Semua library berhasil di-import!')
print('🍠 Siap analisis UMKM Es Ubi Ungu!')

✅ Semua library berhasil di-import!
🍠 Siap analisis UMKM Es Ubi Ungu!


---
## 📋 DATA UTAMA UMKM ES UBI UNGU

Semua data di bawah ini adalah **acuan dasar** untuk seluruh perhitungan.
Ubah angka di sini → semua cell berikutnya otomatis ikut berubah.

### 💡 Penjelasan untuk orang awam:
> Data utama ini ibarat **resep dasar** masakan. Kalau kamu ubah takaran garam, semua rasa ikut berubah. Sama halnya di sini — kalau harga ubi naik, langsung kelihatan pengaruhnya ke keuntungan.

In [2]:
# ════════════════════════════════════════════════════════════════
# 🍠  DATA BAHAN BAKU — UBI UNGU
# ════════════════════════════════════════════════════════════════
harga_ubi_per_kg     = 15000     # Rp/kg  — harga ubi ungu di pasar
kebutuhan_ubi_per_cup = 0.080    # kg/cup — sekitar 80 gram per cup
harga_ubi_per_cup    = harga_ubi_per_kg * kebutuhan_ubi_per_cup  # Rp/cup

# Stok & pembelian ubi
beli_ubi_per_hari_kg = 5.0       # kg/hari — biasanya beli 5 kg/hari
biaya_ubi_per_hari   = harga_ubi_per_kg * beli_ubi_per_hari_kg

# ════════════════════════════════════════════════════════════════
# 🥤  DATA CUP PLASTIK MINUMAN
# ════════════════════════════════════════════════════════════════
harga_cup_per_lusin  = 18000     # Rp/lusin (12 cup)
harga_cup_per_pcs    = harga_cup_per_lusin / 12   # Rp/cup

harga_tutup_per_pcs  = 500       # Rp/pcs (tutup cup)
harga_sedotan_per_pcs = 200      # Rp/pcs
biaya_cup_total_per_pcs = harga_cup_per_pcs + harga_tutup_per_pcs + harga_sedotan_per_pcs

# ════════════════════════════════════════════════════════════════
# 🧂  BAHAN PENDUKUNG LAIN (per cup)
# ════════════════════════════════════════════════════════════════
biaya_gula_per_cup    = 800      # Rp/cup  — gula pasir
biaya_susu_per_cup    = 600      # Rp/cup  — susu kental manis
biaya_es_batu_per_cup = 300      # Rp/cup  — es batu
biaya_gas_per_cup     = 200      # Rp/cup  — alokasi gas untuk merebus
biaya_lain_per_cup    = 200      # Rp/cup  — plastik kresek, tisu, dll

# ════════════════════════════════════════════════════════════════
# 💰  BIAYA VARIABEL TOTAL PER CUP
# ════════════════════════════════════════════════════════════════
modal_per_cup = (
    harga_ubi_per_cup       +
    biaya_cup_total_per_pcs +
    biaya_gula_per_cup      +
    biaya_susu_per_cup      +
    biaya_es_batu_per_cup   +
    biaya_gas_per_cup       +
    biaya_lain_per_cup
)

# ════════════════════════════════════════════════════════════════
# 🏪  DATA PENJUALAN & BIAYA TETAP
# ════════════════════════════════════════════════════════════════
harga_jual           = 10000     # Rp/cup  — harga jual ke konsumen
pendapatan_hari      = 350000    # Rp/hari — target omzet harian
modal_awal           = 120000    # Rp      — biaya tetap/hari (sewa, dll)

# Cup terjual per hari (estimasi dari omzet)
cup_terjual_per_hari = int(pendapatan_hari / harga_jual)

# Biaya tetap per hari (fix cost)
biaya_sewa_per_hari  = 30000     # Rp/hari
biaya_listrik_per_hari = 10000   # Rp/hari
biaya_tenaga_per_hari  = 50000   # Rp/hari (upah bantu)
biaya_lainnya_tetap    = 10000   # Rp/hari (lain-lain tetap)
total_biaya_tetap = biaya_sewa_per_hari + biaya_listrik_per_hari + biaya_tenaga_per_hari + biaya_lainnya_tetap

# Kalkulasi utama
pendapatan_bersih    = pendapatan_hari - modal_awal
keuntungan_per_cup   = harga_jual - modal_per_cup

# ════════════════════════════════════════════════════════════════
# 🫙  DIMENSI CUP/BOTOL (Tabung Terpancung)
# ════════════════════════════════════════════════════════════════
d_atas   = 9.0    # cm — diameter atas cup
d_bawah  = 5.5    # cm — diameter bawah cup
tinggi   = 12.0   # cm — tinggi cup
r1       = d_atas  / 2
r2       = d_bawah / 2

# ════════════════════════════════════════════════════════════════
# 📊  CETAK TABEL RINGKASAN DATA
# ════════════════════════════════════════════════════════════════

print("╔══════════════════════════════════════════════════════════════╗")
print("║         🍠  DATA UTAMA — UMKM ES UBI UNGU                  ║")
print("╚══════════════════════════════════════════════════════════════╝")

print("\n📦  [A] DATA BAHAN BAKU UBI UNGU")
print("-" * 55)
data_ubi = {
    'Item': ['Harga Ubi Ungu', 'Kebutuhan per Cup', 'Biaya Ubi per Cup', 'Pembelian per Hari', 'Biaya Ubi per Hari'],
    'Nilai': [f'Rp {harga_ubi_per_kg:,.0f}/kg', f'{kebutuhan_ubi_per_cup*1000:.0f} gram', 
              f'Rp {harga_ubi_per_cup:,.0f}', f'{beli_ubi_per_hari_kg} kg', 
              f'Rp {biaya_ubi_per_hari:,.0f}'],
    'Keterangan': ['Harga pasar ubi ungu segar', 'Perlu ±80g ubi untuk 1 cup', 
                   'Harga ubi yg masuk 1 cup', 'Total ubi yg dibeli tiap hari', 
                   'Total uang keluar untuk ubi']
}
df_ubi = pd.DataFrame(data_ubi)
print(df_ubi.to_string(index=False))

print("\n🥤  [B] DATA CUP PLASTIK MINUMAN")
print("-" * 55)
data_cup = {
    'Item': ['Harga Cup (per lusin)', 'Harga Cup (per pcs)', 'Tutup Cup', 'Sedotan', 'Total Kemasan/Cup'],
    'Nilai': [f'Rp {harga_cup_per_lusin:,.0f}', f'Rp {harga_cup_per_pcs:,.0f}', 
              f'Rp {harga_tutup_per_pcs:,.0f}', f'Rp {harga_sedotan_per_pcs:,.0f}',
              f'Rp {biaya_cup_total_per_pcs:,.0f}'],
    'Keterangan': ['1 pack isi 12 pcs', 'Rp 18.000 ÷ 12 pcs', 'Tutup seal cup', 'Sedotan plastik', 'Cup + tutup + sedotan']
}
df_cup = pd.DataFrame(data_cup)
print(df_cup.to_string(index=False))

print("\n🧂  [C] BAHAN PENDUKUNG PER CUP")
print("-" * 55)
data_bahan = {
    'Bahan': ['Ubi Ungu', 'Cup + Tutup + Sedotan', 'Gula Pasir', 'Susu Kental Manis', 'Es Batu', 'Gas (alokasi)', 'Lain-lain'],
    'Biaya/Cup (Rp)': [f'{harga_ubi_per_cup:,.0f}', f'{biaya_cup_total_per_pcs:,.0f}', 
                       f'{biaya_gula_per_cup:,.0f}', f'{biaya_susu_per_cup:,.0f}', 
                       f'{biaya_es_batu_per_cup:,.0f}', f'{biaya_gas_per_cup:,.0f}', f'{biaya_lain_per_cup:,.0f}'],
    '% dari Modal': [f'{harga_ubi_per_cup/modal_per_cup*100:.1f}%', f'{biaya_cup_total_per_pcs/modal_per_cup*100:.1f}%',
                     f'{biaya_gula_per_cup/modal_per_cup*100:.1f}%', f'{biaya_susu_per_cup/modal_per_cup*100:.1f}%',
                     f'{biaya_es_batu_per_cup/modal_per_cup*100:.1f}%', f'{biaya_gas_per_cup/modal_per_cup*100:.1f}%',
                     f'{biaya_lain_per_cup/modal_per_cup*100:.1f}%']
}
df_bahan = pd.DataFrame(data_bahan)
print(df_bahan.to_string(index=False))
print("-" * 55)
print(f"  TOTAL MODAL per Cup              : Rp {modal_per_cup:,.0f}")

print("\n💰  [D] BIAYA TETAP HARIAN")
print("-" * 55)
data_tetap = {
    'Item Biaya': ['Sewa tempat', 'Listrik', 'Tenaga kerja', 'Lain-lain tetap', 'TOTAL BIAYA TETAP'],
    'Biaya/Hari (Rp)': [f'{biaya_sewa_per_hari:,.0f}', f'{biaya_listrik_per_hari:,.0f}', 
                        f'{biaya_tenaga_per_hari:,.0f}', f'{biaya_lainnya_tetap:,.0f}',
                        f'{total_biaya_tetap:,.0f}']
}
df_tetap = pd.DataFrame(data_tetap)
print(df_tetap.to_string(index=False))

print("\n🏪  [E] RINGKASAN PENJUALAN")
print("-" * 55)
print(f"  Harga Jual per Cup      : Rp {harga_jual:,.0f}")
print(f"  Modal per Cup           : Rp {modal_per_cup:,.0f}")
print(f"  Keuntungan per Cup      : Rp {keuntungan_per_cup:,.2f}")
print(f"  Target Omzet Harian     : Rp {pendapatan_hari:,.0f}")
print(f"  Estimasi Cup Terjual    : {cup_terjual_per_hari} cup/hari")
print(f"  Margin per Cup          : {keuntungan_per_cup/harga_jual*100:.1f}%")

print("\n🫙  [F] DIMENSI CUP")
print("-" * 55)
print(f"  Diameter Atas  (d₁)     : {d_atas} cm")
print(f"  Diameter Bawah (d₂)     : {d_bawah} cm")
print(f"  Tinggi Cup     (h)      : {tinggi} cm")
print(f"  Jari-jari Atas (r₁)     : {r1} cm")
print(f"  Jari-jari Bawah(r₂)     : {r2} cm")
print("═" * 55)

╔══════════════════════════════════════════════════════════════╗
║         🍠  DATA UTAMA — UMKM ES UBI UNGU                  ║
╚══════════════════════════════════════════════════════════════╝

📦  [A] DATA BAHAN BAKU UBI UNGU
-------------------------------------------------------
              Item        Nilai                    Keterangan
    Harga Ubi Ungu Rp 15,000/kg    Harga pasar ubi ungu segar
 Kebutuhan per Cup      80 gram    Perlu ±80g ubi untuk 1 cup
 Biaya Ubi per Cup     Rp 1,200      Harga ubi yg masuk 1 cup
Pembelian per Hari       5.0 kg Total ubi yg dibeli tiap hari
Biaya Ubi per Hari    Rp 75,000   Total uang keluar untuk ubi

🥤  [B] DATA CUP PLASTIK MINUMAN
-------------------------------------------------------
                 Item     Nilai            Keterangan
Harga Cup (per lusin) Rp 18,000     1 pack isi 12 pcs
  Harga Cup (per pcs)  Rp 1,500    Rp 18.000 ÷ 12 pcs
            Tutup Cup    Rp 500        Tutup seal cup
              Sedotan    Rp 200       Sedo

---
## 📋 TABEL RINGKASAN AWAL — Sebelum Perhitungan Dimulai

> 💡 **Untuk orang awam:** Sebelum kita hitung apa-apa, di bawah ini adalah **semua data modal dan penghasilan** usaha Es Ubi Ungu dalam format tabel yang mudah dibaca. Anggap ini seperti *catatan warung* yang rapi — semua sudah tercatat sebelum kasir mulai menghitung.

---
### 📌 Cara Membaca Tabel Ini:
- **Tabel A** → Apa saja bahan baku dan harganya?
- **Tabel B** → Berapa biaya kemasan (cup) per biji?
- **Tabel C** → Total semua biaya untuk 1 cup Es Ubi Ungu
- **Tabel D** → Biaya tetap yang keluar tiap hari (sewa, listrik, dll)
- **Tabel E** → Penghasilan/pendapatan: berapa yang masuk, berapa yang bersih


---
## 1. 📐 Perhitungan Volume Cup (Integral Benda Putar)

### 1. 📐 Model Matematika
Kita menggunakan metode **Integral Benda Putar** untuk menghitung volume cup (frustum).
Fungsi jari-jari terhadap tinggi $y$:
$$r(y) = r_{bawah} + \left( \frac{r_{atas} - r_{bawah}}{h} \right) y$$
Volume:
$$V = \pi \int_{0}^{h} [r(y)]^2 \, dy$$

### 2. 🧮 Perhitungan Integral & Turunan (Step-by-Step)
Substitusi nilai: $r_1=4.5, r_2=2.75, h=12$
1. Persamaan Jari-jari:
$$r(y) = 2.75 + \frac{1.75}{12}y$$
2. Persamaan Integral:
$$V = \pi \int_{0}^{12} (2.75 + 0.1458y)^2 \, dy$$
3. Ekspansi Polinomial:
$$V = \pi \int_{0}^{12} (7.5625 + 0.8019y + 0.0212y^2) \, dy$$
4. Hasil Integrasi:
$$V = \pi \left[ 7.5625y + 0.4009y^2 + 0.0071y^3 \right]_{0}^{12}$$
$$V = \pi (90.75 + 57.73 + 12.27) = 160.75\pi \approx 505 \text{ ml}$$

### 3. 📊 Analisis Hasil
Berdasarkan perhitungan integral, volume cup adalah **505 ml**. Ini menunjukkan cup ukuran medium-large yang umum di pasaran.

### 4. 🏁 Kesimpulan Optimasi
Volume optimal untuk kemasan Es Ubi Ungu adalah 500-505 ml agar komposisi rasa tetap seimbang antara ubi, susu, dan es batu.

In [3]:
# 📐 KALKULASI VOLUME DENGAN SCIPY
def r_func(y): return r2 + ((r1 - r2) / tinggi) * y
def area_func(y): return np.pi * (r_func(y)**2)
v_int, _ = integrate.quad(area_func, 0, tinggi)
print(f"Hasil Integral: {v_int:.2f} ml")

Hasil Integral: 505.01 ml


---
## 2. 💰 Model Biaya Produksi C(x)

### 1. 📐 Model Matematika (Berdasarkan Integral)
Kita mendefinisikan biaya total $C(x)$ sebagai hasil integral dari **Marginal Cost (MC)**. 
Berdasarkan analisis teknis, setiap penambahan unit produksi memiliki biaya tambahan yang sedikit meningkat (disebabkan kelelahan alat/tenaga).
$$MC = C'(x) = 5.500 + 2x$$
$$C(x) = \int (5.500 + 2x) \, dx$$

### 2. 🧮 Perhitungan Integral (Step-by-Step)
1. Fungsi Integral:
$$C(x) = 5.500x + x^2 + FC$$
2. Dengan Biaya Tetap ($FC$) = Rp 100.000:
$$C(x) = x^2 + 5.500x + 100.000$$

### 3. 📊 Analisis Hasil
Model ini menunjukkan biaya tidak hanya linear, tapi meningkat secara kuadratik. Artinya, memproduksi terlalu banyak dalam satu waktu akan meningkatkan biaya per unit secara eksponensial.

### 4. 🏁 Kesimpulan
Struktur biaya ini lebih realistis untuk menangkap penurunan efisiensi pada skala produksi yang sangat tinggi.

In [4]:
def C(x): 
    # Model sesuai catatan: x^2 + 5500x + 100000
    return (x**2) + (modal_per_cup * x) + total_biaya_tetap

print(f"Biaya produksi 50 cup: Rp {C(50):,.0f}")
print(f"Biaya produksi 100 cup: Rp {C(100):,.0f}")

Biaya produksi 50 cup: Rp 377,500
Biaya produksi 100 cup: Rp 660,000


---
## 3. 📈 Analisis Pendapatan R(x) & Data 1 Minggu

### 1. 📐 Model Matematika
Fungsi Pendapatan $R(x)$ dari hasil penjualan $x$ unit:
$$R(x) = p \cdot x$$
Dimana $p$ adalah harga jual per unit.

### 2. 🧮 Perhitungan Integral & Turunan (Step-by-Step)
Dengan harga jual $p = 10.000$:
1. Fungsi Pendapatan:
$$R(x) = 10.000x$$
2. Jika terjual 1 minggu (total 312 cup):
$$R(312) = 10.000 \cdot 312 = 3.120.000$$

### 3. 📊 Analisis Hasil
Pendapatan meningkat secara linear terhadap jumlah cup yang terjual. Grafik menunjukkan tren positif di akhir pekan.

### 4. 🏁 Kesimpulan Optimasi
Target omzet harian Rp 350.000 (35 cup) sangat realistis dicapai melihat tren data.

In [5]:
# Data simulasi 1 minggu
penjualan = [32, 35, 30, 38, 45, 62, 70]
rev_harian = [x * harga_jual for x in penjualan]
print(f"Total Pendapatan Seminggu: Rp {sum(rev_harian):,.0f}")

Total Pendapatan Seminggu: Rp 3,120,000


---
## 4. 🌿 Model Pertumbuhan Eksponensial

### 1. 📐 Model Matematika
Model pertumbuhan eksponensial untuk memprediksi jumlah cup terjual:
$$N(t) = N_0 e^{kt}$$

### 2. 🧮 Perhitungan Integral & Turunan (Step-by-Step)
Diketahui $N(0) = 32$ (Senin) dan $N(6) = 70$ (Minggu):
1. Persamaan:
$$70 = 32 \cdot e^{k \cdot 6}$$
2. Selesaikan untuk $k$:
$$\frac{70}{32} = e^{6k} \implies 2.1875 = e^{6k}$$
$$\ln(2.1875) = 6k \implies 0.7827 = 6k$$
$$k = 0.1305$$

### 3. 📊 Analisis Hasil
Laju pertumbuhan ($k$) sebesar **13.05%** per hari menunjukkan bisnis berkembang sangat cepat dalam satu minggu pertama.

### 4. 🏁 Kesimpulan Optimasi
Jika tren ini bertahan, diprediksi dalam 2 minggu penjualan akan melampaui 100 cup/hari.

In [6]:
k_val = np.log(70/32) / 6
print(f"Laju Pertumbuhan (k): {k_val:.4f} ({k_val*100:.2f}%)")

Laju Pertumbuhan (k): 0.1305 (13.05%)


---
## 5. 🎯 Optimasi Keuntungan (Maksimum)

### 1. 📐 Model Matematika
Keuntungan (Profit) $\pi(x)$ adalah selisih antara Pendapatan $R(x)$ dan Biaya Total $C(x)$.
$$\pi(x) = R(x) - C(x)$$
$$\pi(x) = 10.000x - (x^2 + 5.500x + 100.000)$$
$$\pi(x) = -x^2 + 4.500x - 100.000$$

### 2. 🧮 Perhitungan Turunan (Mencari Titik Optimal)
Untuk mencari keuntungan maksimal, kita cari nilai $x$ saat turunan pertama sama dengan nol ($\pi'(x) = 0$):
1. Turunan Pertama:
$$\pi'(x) = -2x + 4.500$$
2. Mencari Nilai Optimal:
$$-2x + 4.500 = 0 \implies 2x = 4.500$$
$$x = 2.250 \text{ cup}$$

### 3. 📊 Analisis Hasil
Berdasarkan model biaya kuadratik yang diberikan, titik keuntungan maksimum berada di angka **2.250 cup**. Namun, untuk skala UMKM, angka ini merupakan batas kapasitas teoritis yang sangat besar.

### 4. 🏁 Kesimpulan Optimasi
Selama produksi harian masih di bawah 2.250 cup, setiap penambahan produksi akan terus meningkatkan total keuntungan.

In [7]:
# Mencari x optimal: pi'(x) = 4500 - 2x = 0
x_opt = 4500 / 2
print(f"Volume Produksi Optimal (Teoritis): {int(x_opt)} cup")

# Visualisasi sederhana profit pada rentang UMKM (0-100 cup)
def profit(x): return (harga_jual * x) - C(x)
print(f"Profit pada 50 cup: Rp {profit(50):,.0f}")
print(f"Profit pada 100 cup: Rp {profit(100):,.0f}")

Volume Produksi Optimal (Teoritis): 2250 cup
Profit pada 50 cup: Rp 122,500
Profit pada 100 cup: Rp 340,000


---
## 6. 🧩 Bonus: Contoh Integral Parsial (Biaya Kompleks)

### 1. 📐 Model Matematika
Digunakan untuk fungsi yang merupakan perkalian dua variabel (u dan v):
$$\int u \, dv = uv - \int v \, du$$

### 2. 🧮 Perhitungan Integral & Turunan (Step-by-Step)
Misalkan fungsi akumulasi biaya operasional harian: $\int t e^{-0.1t} dt$
1. Tentukan $u$ dan $dv$:
$$u = t \implies du = dt$$
$$dv = e^{-0.1t} dt \implies v = -10e^{-0.1t}$$
2. Gunakan Rumus:
$$\int t e^{-0.1t} dt = t(-10e^{-0.1t}) - \int (-10e^{-0.1t}) dt$$
$$-10t e^{-0.1t} - 100e^{-0.1t} + C$$

### 3. 📊 Analisis Hasil
Integral ini membantu menghitung total akumulasi biaya yang mengalami penurunan efisiensi seiring berjalannya waktu.

### 4. 🏁 Kesimpulan Optimasi
Metode integral parsial terbukti efektif untuk pemodelan biaya jangka panjang yang melibatkan variabel waktu dan peluruhan (decay).

In [8]:
t_sym = sp.symbols('t')
print("Hasil Integral Parsial:")
display(sp.integrate(t_sym * sp.exp(-0.1*t_sym), t_sym))

Hasil Integral Parsial:


1.0*(-10.0*t - 100.0)*exp(-0.1*t)